In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import col, count, when, isnan, isnull, trim, length, current_timestamp, from_utc_timestamp, row_number, udf, lit, regexp_replace
from pyspark.sql.types import StringType, BooleanType
from pyspark.sql.window import Window


def calcular_completude(df: DataFrame, colunas_obrigatorias: list) -> dict:
    total_registros = df.count()
    metricas = {}
    for coluna in colunas_obrigatorias:
        nao_nulos = df.filter(col(coluna).isNotNull()).count()
        taxa_completude = (nao_nulos / total_registros * 100) if total_registros > 0 else 0
        metricas[coluna] = {
            'total': total_registros,
            'preenchidos': nao_nulos,
            'nulos': total_registros - nao_nulos,
            'taxa_completude_%': round(taxa_completude, 2)
        }
    return metricas

def validar_precisao_numerica(df: DataFrame, coluna: str, min_val: float = None, max_val: float = None) -> DataFrame:
    condicao = col(coluna).isNotNull()
    if min_val is not None:
        condicao = condicao & (col(coluna) >= min_val)
    if max_val is not None:
        condicao = condicao & (col(coluna) <= max_val)
    return df.filter(condicao)

def remover_duplicados(df: DataFrame, chave_primaria: list, criterio_desempate: str = "hora_ingestao") -> DataFrame:
    window_spec = Window.partitionBy(chave_primaria).orderBy(col(criterio_desempate).desc())
    df_deduplicated = df.withColumn("row_num", row_number().over(window_spec)) \
                        .filter(col("row_num") == 1) \
                        .drop("row_num")
    return df_deduplicated

def adicionar_metadados_silver(df: DataFrame) -> DataFrame:
    return df.withColumn("data_processamento_silver", 
                         from_utc_timestamp(current_timestamp(), "America/Sao_Paulo"))


In [0]:

# COLOCAR LIMITE DA BASE SE QUISER AMOSTRA ==============================================================================================================

# Configuração
CATALOG = "workspace"
SCHEMA_BRONZE = "yelp_bronze"
SCHEMA_SILVER = "yelp_silver"

# ========== TABELA 3: USER ==========


table_name = "yelp_academic_dataset_user"
print(f"Processando tabela: {table_name}")
print("="*60)

# Leitura da tabela bronze
df_user = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.{table_name}")
print(f"Registros originais: {df_user.count()}")

# Carrega tabelas review já processada
table_new_silver_review = f"{CATALOG}.{SCHEMA_SILVER}.review"
df_review = spark.table(table_new_silver_review)

# Carrega tabelas silver
print(f"Total de usuários: {df_user.count()}")

# Carrega user_ids presentes silver review
user_ids_food = spark.table(table_new_silver_review).select("user_id").distinct()

# Filtra df_user apenas com user_ids presentes em silver review -> usuários que fizeram reviews em estabelecimentos das categorias de interesse
df_users_food = df_user.join(user_ids_food, on="user_id", how="inner")

print(f"Total de usuários com review de food: {df_users_food.count()}")

# Remove colunas que não serão utilizadas da df_users_food
cols_to_drop = [
    "compliment_cool", "compliment_cute", "compliment_funny", "compliment_hot",
    "compliment_list", "compliment_more", "compliment_note", "compliment_photos",
    "compliment_plain", "compliment_profile", "compliment_writer", "cool",
    "friends", "funny","useful"
]
df_users_food_clean = df_users_food.drop(*cols_to_drop)

# COMPLETUDE: Campos obrigatórios
colunas_obrigatorias = ['user_id', 'name']
metricas_completude = calcular_completude(df_users_food_clean, colunas_obrigatorias)

print("\n--- Métricas de COMPLETUDE ---")
for col_name, metricas in metricas_completude.items():
    print(f"  {col_name}: {metricas['taxa_completude_%']}% completo ({metricas['nulos']} nulos)")

# Filtra registros com campos obrigatórios preenchidos
df_users_food_clean = df_users_food_clean.filter(
    col('user_id').isNotNull() & 
    col('name').isNotNull()
)

print(f"\nApós filtro de completude: {df_users_food_clean.count()} registros")

# PRECISÃO: Validações numéricas
print("\n--- Validação de PRECISÃO ---")

# Review count, fans
for coluna in ['review_count', 'fans']:
    df_users_food_clean = df_users_food_clean.filter(
        col(coluna).isNull() | (col(coluna) >= 0)
    )
print(f"  Contadores (>=0): {df_users_food_clean.count()} registros válidos")

# Remoção de duplicados
print("\n--- Remoção de Duplicados ---")
df_users_food_clean = remover_duplicados(df_users_food_clean, ['user_id'])
print(f"Após deduplicação: {df_users_food_clean.count()} registros")

## Corrige o erro de formatação na coluna elite onde '2020' aparece como '20,20'
df_users_food_clean = df_users_food_clean.withColumn(
        "elite",
        regexp_replace(col("elite"), "20,20", "2020")
    )
print(f"Coluna Elite corrigida.")


# ========== ADICIONA CAMPO review_food_count: contagem de reviews para estabelecimentos  de interesse ==========
print("\n--- Adicionando campo review_food_count ---")

# Calcula review_food_count por user_id
df_review_food_count = df_review.groupBy("user_id").agg(
    count("review_id").alias("review_food_count")
)

# LEFT JOIN: adiciona review_food_count (0 se não tiver reviews)
df_users_food_clean = df_users_food_clean.join(
    df_review_food_count,
    on="user_id",
    how="left"
).withColumn(
    "review_food_count",
    when(col("review_food_count").isNull(), lit(0)).otherwise(col("review_food_count"))
)

users_with_food = df_users_food_clean.filter(col("review_food_count") > 0).count()
users_without_food = df_users_food_clean.filter(col("review_food_count") == 0).count()
print(f"  Usuários com reviews food: {users_with_food}")
print(f"  Usuários sem reviews food: {users_without_food}")

# Adiciona metadados silver
df_user_silver = adicionar_metadados_silver(df_users_food_clean)

# Salva na camada Silver
table_silver = f"{CATALOG}.{SCHEMA_SILVER}.user"
df_user_silver.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_silver)

print(f"\n✓ Tabela silver criada: {table_silver}")
print(f"Total de registros silver: {df_user_silver.count()}")
print("="*60)

In [0]:
# Visualiza amostra dos dados limpos
print("AMOSTRA - Tabela Silver User:")
print("="*60)

df_sample = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.user")

print(f"Total de reviews: {df_sample.count()}")

# Amostra de dados
print("\nPrimeiros 5 registros:")
display(df_sample.select(
    'user_id',
    'name',
    'average_stars',
    'elite',
    'fans',
    'review_count',
    'yelping_since',
    'review_food_count',
    'data_processamento_silver',
    'hora_ingestao'
).limit(5))


# Distribuição de stars
print("\nDistribuição de Stars:")
display(df_sample.groupBy('average_stars').count().orderBy('average_stars'))